In [ ]:
import pandas as pd 
df = pd.read_csv("train.csv")
display(df.head());
print(df.shape)

In [ ]:
df.describe()

In [ ]:
df.drop("id", axis=1)

In [ ]:
df.dtypes

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# X = df.drop(columns=["Will_Buy_EV"])
# y = df["Will_Buy_EV"]

X = pd.get_dummies(df.drop(columns=["Will_Buy_EV", "id"]), drop_first=True)
y = df["Will_Buy_EV"].map({"Yes": 1, "No": 0})

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
pred = model.predict_proba(X_valid)[:, 1]
print(roc_auc_score(y_valid, pred))

* Now let's try to perform standardization here as 6 of the columns are categorical.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict_proba(X_valid)[:, 1]

print(roc_auc_score(y_valid, pred))

* Accuracy Improved from 0.90 to 0.93
## Let's try using XGBoost now

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

X = pd.get_dummies(df.drop(columns=["Will_Buy_EV", "id"]), drop_first=True)
y = df["Will_Buy_EV"].map({"Yes": 1, "No": 0})

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict_proba(X_valid)[:, 1]

print(roc_auc_score(y_valid, pred))

* Reached 0.90 Accuracy -> 0.93 (Using Standardization) -> 0.94 (Using XGBoost)
* Let's try CatBoost now :

In [ ]:
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score

model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    random_seed=42,
    verbose=False
)

model.fit(X_train, y_train)

pred = model.predict_proba(X_valid)[:, 1]

print(roc_auc_score(y_valid, pred))

* CatBoost reaches a 0.0008 lower accuracy than XGBoost, which is insignificant in hindsight.
* Before moving forward let's try plotting some graphs or subplots and check out how the data is distributed.

In [ ]:
%pip install seaborn
import seaborn as sns
import matplotlib.pyplot as plt

sns.boxplot(data=df, x="Will_Buy_EV", y="Age")
plt.show()

In [ ]:
sns.boxplot(data=df, x="Will_Buy_EV", y="Annual_Income_USD")
plt.show()

In [ ]:
sns.boxplot(
    data=df,
    x="Will_Buy_EV",
    y="Charging_Stations_Near_Home"
)
plt.show()

* Plots really didn't tell me anything, the dataset is pretty evenly distributed apart from the fact that low-mid earning individuals don't prefer purchasing EV's.
* Let's try to get into more detail for this using some plots.

In [ ]:
pd.crosstab(
    pd.qcut(df["Annual_Income_USD"], 5),
    df["Will_Buy_EV"],
    normalize="index"
)

In [ ]:
df.groupby("Will_Buy_EV")["Annual_Income_USD"].mean()

* Income feels highly predictive of the final result now, it keeps increasing as we go up and seems the strongest feature in our dataframe.

In [ ]:
sns.boxplot(data=df, x="Will_Buy_EV", y="Annual_Income_USD")

In [ ]:
pd.crosstab(df["Subsidy_Available"], df["Will_Buy_EV"], normalize="index")

* We've just realized, subsidy available is a incredibly strong feature as well, maybe even stronger than Income.
* Let's try to build and train our better baseline model now

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1}) # just binary classification 
X = train.drop(columns=["Will_Buy_EV"])
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    verbose=50,
    random_seed=42
)

model.fit(
    X_train,
    y_train,
    cat_features=cat_cols
)

predictions = model.predict_proba(X_valid)[:, 1]
score = roc_auc_score(y_valid, predictions)
print("ROC-AUC:", score)

* Okay 0.941 is good enough for my first proper baseline model, the leaderboard top is 0.946 so I'm far by 0.005.
* Let's make this submission and then try newer techniques later.

In [ ]:
# Preparing a submission.csv for submission now.
test_predictions = model.predict_proba(test)[:, 1]
submission = pd.DataFrame({
    "id": test["id"],
    "Will_Buy_EV": test_predictions
})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print(submission.shape)

* Let's run k-fold now on the training dataset and then check results and optimize the hyperparameters.


In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")

y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})
X = train.drop(columns=["Will_Buy_EV"])

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), 1):
    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=500,
        learning_rate=0.05,
        depth=6,
        verbose=False,
        random_seed=42
    )

    model.fit(
        X_train,
        y_train,
        cat_features=cat_cols
    )

    predictions = model.predict_proba(X_valid)[:, 1]
    score = roc_auc_score(y_valid, predictions)

    scores.append(score)

    print(f"Fold {fold}: {score:.6f}")

print("\nMean ROC-AUC:", sum(scores) / len(scores))
print("Std ROC-AUC:", pd.Series(scores).std())

* Final baseline model set at accuracy 0.941, let the hyperparameters tuning begin!

# Hyperparameter Tuning

## 1. Depth

| Depth | ROC-AUC |
|------:|--------:|
| 4     | 0.941000 |
| 6     | 0.941067 |
| 8     | **0.941125** |
| 10    | 0.941009 |
| 11    | 0.941096 |
| 12    | 0.940486 |

The ROC-AUC peaked at **depth = 8**, so we'll pick this as our depth hyperparameter.

---

## 2. Learning Rate

| Hyperparameter | ROC-AUC |
|------:|--------:|
| 0.01     | 0.9397632329260461 |
| 0.03     | 0.9406599579823963 |
| 0.05     | 0.941125 |
| 0.1     |  **0.9412652935364746** |
| 0.2    | 0.9402401598427775 |

The ROC-AUC peaked at **Learning Rate = 0.1**, so we'll pick this as our learning rate hyperparameter.

---


## 3. L2 Regularization

| Hyperparameter | ROC-AUC |
|------:|--------:|
| 1     | 0.9400649805975736 |
| 3     | **0.9412652935364746** |
| 5     | 0.9403550589882825 |
| 10     | 0.9405684202847963  |

The ROC-AUC peaked at **Learning Rate = 0.1**, so we'll pick this as our learning rate hyperparameter.

---

## 4. Iterations

| Hyperparameter | ROC-AUC |
|------:|--------:|
| 300     | **0.9406698297215036** |
| 500     | 0.9399540407762492 |
| 800     | 0.9389406070076945  |
| 1200     | 0.937831279004862  |
| 2000     | 0.9358642051373034  |

The ROC-AUC peaked at **Iterations = 0.1**, so we'll pick this as our iterations hyperparameter.

---

## 5. Random Strength

| Hyperparameter | ROC-AUC |
|------:|--------:|
| 1     | 0.9400649805975736 |
| 2     | 0.9401554965591212 |

The ROC-AUC peaked at **Random Strength = 2**, so we'll pick this as our Random Strength hyperparameter.

---

## 5. Bagging Temperature

| Hyperparameter | ROC-AUC |
|------:|--------:|
| 0     | 0.9400649805975736 |
| 0.5   | 0.9400649805975736 |
| 1     | 0.9400649805975736 |
| 2     | 0.9400649805975736 |

The ROC-AUC peaked at **Bagging Temperature = 0**, so we'll pick this as our Bagging Temperature hyperparameter.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")

y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})
X = train.drop(columns = ["Will_Buy_EV"])

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 42,
    stratify = y
)

model = CatBoostClassifier(
    iterations = 300,
    learning_rate = 0.1,
    depth = 8,
    thread_count=-1,
    l2_leaf_reg = 3,
    verbose = 100,
    random_seed = 42,
    bagging_temperature=0,
    random_strength=2,
    early_stopping_rounds=100
)

model.fit(
    X_train,
    y_train,
    cat_features = cat_cols
)

predictions = model.predict_proba(X_valid)[:, 1]
score = roc_auc_score(y_valid, predictions)
print("ROC-AUC:", score)

* After testing on the tuned hyperparameters an accuracy of 0.941076 is achieved.
* Let's create a submission.csv and submit it for now.

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})

X = train.drop(columns=["Will_Buy_EV"])

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=8,
    thread_count=-1,
    l2_leaf_reg=3,
    verbose=100,
    random_seed=42,
    bagging_temperature=0,
    random_strength=2
)

model.fit(
    X,
    y,
    cat_features=cat_cols
)

test_predictions = model.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    "id": test["id"],
    "Will_Buy_EV": test_predictions
})

submission.to_csv("submission.csv", index=False)

print(submission.head())
print("Submission shape:", submission.shape)

# Feature Engineering & Model Experiments (post-0.94083 submission)

* I've now added AI in the loop to learn more insights on unexplored topics.
* Systematic experiment log, all measured against the tuned CatBoost config above that scored **0.94083** on the public leaderboard (local holdout: 0.9410766, 5-fold CV: 0.9412667).

All code lives in `experiments/` as standalone, reproducible scripts (not squeezed into
this notebook, since some fits take minutes each on 668K rows) see `experiments/results.jsonl`
for the raw log of every run.

## Critical finding: `id` was leaking train/test split membership

Both the original baseline above and my first pass of experiments left the `id` column in
`X` as a plain numeric feature. **Adversarial validation** (train a classifier to tell train
rows from test rows) scored **AUC ~1.0**, and `id` alone carried ~96% of the feature
importance for that separation, because test ids (668665+) are a disjoint, strictly higher
range than train ids (0-668664). CatBoost could freely split on `id` thresholds, learning
nothing generalizable and risking overfit to whatever weak correlations exist between row
order and target by chance.

Dropping `id` and re-running adversarial validation: AUC fell to **0.4992** (i.e. ~0.5,
random guessing), confirming there is no genuine train/test distribution drift once the
leaked identifier is removed, and that local CV should track the leaderboard reliably.

Effect size, decomposed with 5-fold CV (isolating variables one at a time, same tuned params):

| Feature set | 5-fold CV |
|---|---:|
| Base features + `id` (apples-to-apples baseline) | 0.941337 |
| Base features, `id` dropped | 0.941395 |

Only **+0.00006** on average across folds, smaller than it looked from a single holdout
split (+0.00114 there), which is itself a useful lesson: a single holdout can overstate an
effect that 5-fold CV shows is mostly noise. Still correct to remove `id`: it has zero
possible predictive value and only downside risk.

## Feature engineering ideas tested (holdout screen, `id` dropped)

Each idea was tested individually, additive-only on top of the base feature set, so any
delta is attributable to that one idea:

| Idea | Hypothesis | Holdout AUC | Delta vs base |
|---|---|---:|---:|
| `Income_x_Subsidy` (income x subsidy flag) | subsidy's effect on purchase probability isn't additive with income -- it may swing a middle-income buyer far more than someone already affluent | 0.941212 | **+0.00099** |
| `Range_Anxiety_Ordinal` (Low/Medium/High -> 0/1/2) | this is genuinely ordinal; giving CatBoost a numeric ordinal instead of an unordered category lets it use `<=` threshold splits instead of set-membership splits | 0.941173 | +0.00096 |
| `City_Type` x `Home_Charging_Possible` combined categorical | "can charge at home" means something different in Urban vs. Rural/Suburban; combining exposes that context in one categorical instead of needing both features split in the same tree path | 0.941169 | +0.00095 |
| `Total_Charging_Stations` + ratio to commute distance | infrastructure access relative to how much you actually drive should matter more than raw counts | 0.941111 | +0.00003 |
| `Concern_minus_Anxiety` (environmental concern minus range anxiety) | high concern + high anxiety should partially cancel out as a purchase driver | 0.941094 | ~flat |
| `Income_per_Car`, `Commute_to_Income` (affordability ratios) | per-person affordability and commute burden relative to income as wealth/need proxies | 0.941093 | ~flat |
| All six combined | -- | 0.941207 | +0.00113 (no better than the single best feature -- see note below) |

**Why "all combined" didn't beat the single best feature:** CatBoost trees already learn
arbitrary interactions/thresholds given enough depth and iterations, so most hand-crafted
interaction features don't add information the model couldn't already reconstruct.
They mostly help when they either (a) need more depth than the model is currently using to
re-derive the pattern, or (b) collapse a ratio the model would otherwise approximate with
many splits. Adding six features at once, several redundant with what CatBoost already
captures, just spends iteration/depth budget without net benefit. This matches why
`Income_x_Subsidy` and the ordinal encoding (both genuinely new information) were the two
that stuck.

## Alternative models tried

| Model | Holdout AUC | Note |
|---|---:|---|
| CatBoost (tuned, same holdout) | 0.941077 | baseline for comparison |
| XGBoost, native categorical (`enable_categorical=True`) | 0.940153 | still behind CatBoost even with native cat splits, not just one-hot |
| CatBoost + XGBoost blend, 50/50 | 0.941122 | |
| CatBoost + XGBoost blend, best weight (0.7 CatBoost / 0.3 XGBoost) | 0.941224 | marginal gain over CatBoost alone, at 2x training/inference cost -- not worth the complexity here |

CatBoost's native categorical handling remains the strongest single model. The blend adds
a small amount but not enough to justify maintaining two models for this dataset size/signal.

## Hyperparameter re-tune on the winning feature set

The original tuning (depth=8, lr=0.1, iterations=300) was done on the `id`-contaminated
feature set and no longer the local optimum once `id` is dropped and the two new features
are added:

| depth | lr | l2 | iterations | Holdout AUC |
|---:|---:|---:|---:|---:|
| 8 | 0.1 | 3 | 300 (old config) | 0.941341 |
| **6** | **0.1** | **3** | **500** | **0.941546** |
| 8 | 0.1 | 5 | 500 | 0.941467 |
| 8 | 0.05 | 3 | 800 | 0.941376 |
| 9 | 0.05 | 3 | 800 | 0.941343 |
| 10 | 0.05 | 5 | 800 | 0.941181 |

Shallower (depth=6) with more iterations at the same learning rate won, consistent with
having removed a spuriously-splittable feature (`id`) that likely encouraged deeper trees
to "use up" capacity on noise.

## Final result

**Pipeline:** base features (`id` dropped) + `Income_x_Subsidy` + `Range_Anxiety_Ordinal`,
CatBoost with `depth=6, learning_rate=0.1, l2_leaf_reg=3, iterations=500`.

| | 5-fold CV |
|---|---:|
| Original submission (scored 0.94083 on LB) | 0.941267 |
| **New pipeline** | **0.941841** |

**+0.00057**, confirmed with 5-fold CV (not just a single holdout), decomposed feature by
feature so the source of the gain is understood rather than just chased. Modest but real
and reproducible; see `experiments/train_final_and_submit.py` to regenerate
`submission_v2.csv` from scratch, and `experiments/results.jsonl` for every run's raw score.

### What didn't pay off (worth recording so it isn't re-tried blindly)
- Charging-infrastructure ratio features: CatBoost already captures this via `Charging_Stations_Near_Home`/`Near_Work` splits.
- Environmental-concern/range-anxiety cancellation feature: flat.
- Affordability ratios (`Income_per_Car`, `Commute_to_Income`): flat.
- XGBoost as a standalone model: consistently behind CatBoost on this data.
- Throwing all feature ideas in at once: worse than the two winners alone (dilutes iteration budget).


In [ ]:
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})
X = train.drop(columns=["Will_Buy_EV", "id"])   
test_X = test.drop(columns=["id"])

def add_features(df):
    df = df.copy()
    df["Income_x_Subsidy"] = df["Annual_Income_USD"] * (df["Subsidy_Available"] == "Yes").astype(int)
    df["Range_Anxiety_Ordinal"] = df["Range_Anxiety_Level"].map({"Low": 0, "Medium": 1, "High": 2})
    return df

X = add_features(X)
test_X = add_features(test_X)

cat_cols = X.select_dtypes(include=["object", "str"]).columns.tolist()

# 5-fold CV confirmation of the final pipeline
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = []
for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y), 1):
    model = CatBoostClassifier(
        depth=6, learning_rate=0.1, l2_leaf_reg=3, iterations=500,
        random_seed=42, thread_count=-1, verbose=False,
    )
    model.fit(X.iloc[train_idx], y.iloc[train_idx], cat_features=cat_cols)
    pred = model.predict_proba(X.iloc[valid_idx])[:, 1]
    scores.append(roc_auc_score(y.iloc[valid_idx], pred))
    print(f"Fold {fold}: {scores[-1]:.6f}")

print("\nMean ROC-AUC:", sum(scores) / len(scores))
print("Std ROC-AUC:", pd.Series(scores).std())

In [ ]:
final_model = CatBoostClassifier(
    depth=6, learning_rate=0.1, l2_leaf_reg=3, iterations=500,
    random_seed=42, thread_count=-1, verbose=100,
)
final_model.fit(X, y, cat_features=cat_cols)

test_predictions = final_model.predict_proba(test_X)[:, 1]
submission_v2 = pd.DataFrame({"id": test["id"], "Will_Buy_EV": test_predictions})
submission_v2.to_csv("submission_v2.csv", index=False)

print(submission_v2.head())
print("Submission shape:", submission_v2.shape)

# Round 3: digit decomposition + LightGBM blend

Prompted by a [public notebook](https://www.kaggle.com/competitions/playground-series-s6e9/discussion/741117) on this exact competition (0.94590 LB) crediting three
techniques: digit decomposition, frequency encoding, and target encoding. Tested all
three individually on top of the round-2 pipeline (same holdout, additive-only):

| Technique | Holdout delta |
|---|---:|
| **Digit decomposition** of `Annual_Income_USD`, `Age`, `Daily_Commute_km` | **+0.00168** |
| Frequency encoding of all categoricals | -0.00003 (flat) |
| Out-of-fold smoothed target encoding of all categoricals | -0.00003 (flat) |
| LightGBM (in place of CatBoost) | +0.00032 |

Frequency/target encoding came back flat here, consistent with the round-2 finding that
CatBoost's own ordered target statistics already capture what an external categorical
encoding would add; there was nothing left on the table there. Digit decomposition is a
different story entirely.

## Why digit decomposition worked

Kaggle Playground Series datasets are synthetically generated, and generators sometimes
apply digit-level rules (e.g. a rounding or bucketing decision keyed on a specific digit
place of a numeric column) rather than smooth functions of the raw magnitude. A smooth
value like `Annual_Income_USD` can't expose that to a tree no matter how deep it goes,
because a digit-level rule isn't monotonic or smooth in the raw value, but splitting the
number into its individual digit-place columns lets the tree split directly on the digit
that the generator actually used.

Feature importance on the fitted model confirms this is real, not spurious:

| Feature | Importance |
|---|---:|
| `Annual_Income_USD_digit_3` (thousands place) | 4.15 |
| `Annual_Income_USD_digit_2` (hundreds place) | 3.57 |
| `Annual_Income_USD_digit_4` (ten-thousands place) | 2.51 |
| `Annual_Income_USD_digit_1` (tens place) | 2.20 |
| `Annual_Income_USD_digit_0` / `_digit_5` | 1.51 / 1.04 |
| `Age_digit_0`, `Daily_Commute_km_digit_0` | 1.66 / 1.44 (small but non-zero) |
| `Age_digit_1..5`, `Commute_digit_1..5` | mostly 0 (dead -- these numbers don't have that many real digits) |

Collectively, `Annual_Income_USD`'s digit columns (~15 importance) outweigh the raw
`Annual_Income_USD` column itself (6.4), strong evidence the data generator encoded
something in income's digits specifically, not just its magnitude. Age and Commute digits
beyond the ones place are structurally always zero (max values are 2-digit numbers) and
were pruned.

5-fold CV confirms this isn't a holdout fluke: **0.941841 -> 0.943210** (+0.00137).

## Re-tuning + LightGBM blend on the expanded feature set

With the dead digit columns pruned, CatBoost's optimum shifted again (slightly deeper,
slightly slower learning rate: depth=7, lr=0.07, l2=4, iterations=700). LightGBM, tried
fresh on this feature set, edged out CatBoost for the first time in this project
(0.943652 vs 0.943385 holdout), and blending the two (0.3 CatBoost / 0.7 LightGBM,
weight chosen on holdout) beat both individually.

5-fold CV, with the blend properly re-generated per fold (each model retrained per fold,
predictions blended on that fold's held-out rows, not just reusing a single holdout's
optimal weight) to avoid overstating the blend's benefit:

| | 5-fold CV |
|---|---:|
| CatBoost (depth=7, lr=0.07, l2=4, iter=700) | 0.943590 |
| LightGBM (depth=7, lr=0.07, iter=600) | 0.943598 |
| **Blend (0.3 CatBoost / 0.7 LightGBM)** | **0.943756** |

## Cumulative result

| Stage | 5-fold CV |
|---|---:|
| Original submission (0.94083 on LB) | 0.941267 |
| Round 2 (leakage fix + 2 features + retune) | 0.941841 |
| **Round 3 (+ digit decomposition + LightGBM blend)** | **0.943756** |

**+0.00249 total.** See `experiments/train_final_v3_and_submit.py` for the full, from-scratch
reproduction (`submission_v3.csv`).

### What's still flat (don't re-try blindly)
- Frequency encoding and out-of-fold target encoding of categoricals: CatBoost's native
  handling already covers this ground.
- Standalone XGBoost: consistently the weakest of the three GBM libraries tried here.

### Natural next steps if pushing further
- Digit decomposition of other numeric columns not yet tried this way (e.g. `Charging_Stations_Near_Home/Work`, though these are already small integers with limited digit range).
- A small logistic-regression or ridge stacking meta-model over CatBoost + LightGBM out-of-fold predictions, instead of a fixed blend weight.
- Optuna/systematic search instead of the small manual grids used here, now that the feature set is more settled.

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

y = train["Will_Buy_EV"].map({"No": 0, "Yes": 1})

def add_features(df):
    df = df.copy()
    df["Income_x_Subsidy"] = df["Annual_Income_USD"] * (df["Subsidy_Available"] == "Yes").astype(int)
    df["Range_Anxiety_Ordinal"] = df["Range_Anxiety_Level"].map({"Low": 0, "Medium": 1, "High": 2})
    for col in ["Annual_Income_USD", "Age", "Daily_Commute_km"]:
        int_part = df[col].fillna(0).astype("int64").abs()
        for place, divisor in enumerate([1, 10, 100, 1000, 10000, 100000]):
            df[f"{col}_digit_{place}"] = (int_part // divisor) % 10
    return df

X = add_features(train.drop(columns=["Will_Buy_EV", "id"]))
test_X = add_features(test.drop(columns=["id"]))

dead_cols = [c for c in X.columns if "_digit_" in c and X[c].nunique() <= 1]
X = X.drop(columns=dead_cols)
test_X = test_X.drop(columns=dead_cols)

cat_cols = X.select_dtypes(include=["object", "str"]).columns.tolist()

CB_PARAMS = dict(depth=7, learning_rate=0.07, l2_leaf_reg=4, iterations=700, random_seed=42, thread_count=-1, verbose=100)
BLEND_W_CB = 0.3 

print("Training CatBoost...")
cb = CatBoostClassifier(**CB_PARAMS)
cb.fit(X, y, cat_features=cat_cols)
cb_test_pred = cb.predict_proba(test_X)[:, 1]

print("Training LightGBM...")
X_lgb, test_X_lgb = X.copy(), test_X.copy()
for c in cat_cols:
    X_lgb[c] = X_lgb[c].astype("category")
    test_X_lgb[c] = test_X_lgb[c].astype("category")
lgb = LGBMClassifier(n_estimators=600, max_depth=7, learning_rate=0.07, random_state=42, verbose=-1)
lgb.fit(X_lgb, y, categorical_feature=cat_cols)
lgb_test_pred = lgb.predict_proba(test_X_lgb)[:, 1]

blend_pred = BLEND_W_CB * cb_test_pred + (1 - BLEND_W_CB) * lgb_test_pred
submission_v3 = pd.DataFrame({"id": test["id"], "Will_Buy_EV": blend_pred})
submission_v3.to_csv("submission_v3.csv", index=False)

print(submission_v3.head())
print("Submission shape:", submission_v3.shape)